In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS spotify_etl.raw.spotify_tokens (
#   token_name STRING,
#   access_token STRING,
#   refresh_token STRING,
#   expires_at BIGINT,
#   updated_at TIMESTAMP
# )
# USING delta;

In [0]:
from pyspark.sql import functions as F
import requests
import base64
import time

TOKEN_TABLE = "spotify_etl.raw.spotify_tokens"
TOKEN_NAME = "default"

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

RAW_TABLE = "spotify_etl.raw.spotify_api_calls"

RAW_SCHEMA = StructType([
    StructField("ingestion_ts", TimestampType(), False),
    StructField("run_id", StringType(), False),
    StructField("token_name", StringType(), True),
    StructField("method", StringType(), False),
    StructField("request_url", StringType(), False),
    StructField("request_params_json", StringType(), True),
    StructField("request_body_json", StringType(), True),
    StructField("http_status", IntegerType(), True),
    StructField("response_headers_json", StringType(), True),
    StructField("payload_json", StringType(), True),
    StructField("error", StringType(), True),
    StructField("attempt", IntegerType(), True),
    StructField("request_hash", StringType(), False),
])

def save_raw_api_call(raw_row: dict) -> None:
    row = {
        "ingestion_ts": None,
        "run_id": str(raw_row.get("run_id") or ""),
        "token_name": raw_row.get("token_name"),
        "method": str(raw_row.get("method") or ""),
        "request_url": str(raw_row.get("request_url") or ""),
        "request_params_json": raw_row.get("request_params_json"),
        "request_body_json": raw_row.get("request_body_json"),
        "http_status": int(raw_row["http_status"]) if raw_row.get("http_status") is not None else None,
        "response_headers_json": raw_row.get("response_headers_json"),
        "payload_json": raw_row.get("payload_json"),
        "error": raw_row.get("error"),
        "attempt": int(raw_row["attempt"]) if raw_row.get("attempt") is not None else None,
        "request_hash": str(raw_row.get("request_hash") or ""),
    }
    df = spark.createDataFrame([Row(**row)], schema=RAW_SCHEMA).withColumn("ingestion_ts", F.current_timestamp())
    df.write.format("delta").mode("append").saveAsTable(RAW_TABLE)

def load_token_state(token_name: str = TOKEN_NAME) -> dict:
    df = spark.table(TOKEN_TABLE).where(F.col("token_name") == token_name)
    rows = df.limit(1).collect()

    if not rows:
        return {"token_name": token_name, "access_token": None, "refresh_token": None, "expires_at": 0}

    r = rows[0].asDict()
    return {
        "token_name": token_name,
        "access_token": r.get("access_token"),
        "refresh_token": r.get("refresh_token"),
        "expires_at": int(r.get("expires_at") or 0)
    }

def save_token_state(state: dict, token_name: str = TOKEN_NAME) -> None:
    # upsert atomic: 1 rând per token_name
    spark.createDataFrame([{
        "token_name": token_name,
        "access_token": state.get("access_token"),
        "refresh_token": state.get("refresh_token"),
        "expires_at": int(state.get("expires_at") or 0),
        "updated_at": None
    }]).withColumn("updated_at", F.current_timestamp()) \
      .createOrReplaceTempView("incoming_token")

    spark.sql(f"""
      MERGE INTO {TOKEN_TABLE} t
      USING incoming_token s
      ON t.token_name = s.token_name
      WHEN MATCHED THEN UPDATE SET
        t.access_token = s.access_token,
        t.refresh_token = s.refresh_token,
        t.expires_at   = s.expires_at,
        t.updated_at   = s.updated_at
      WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
class SpotifyAuthError(RuntimeError):
    pass

def refresh_spotify_access_token(refresh_token: str, client_id: str, client_secret: str, timeout_s: int = 10) -> dict:
    token_url = "https://accounts.spotify.com/api/token"
    basic = base64.b64encode(f"{client_id}:{client_secret}".encode()).decode()

    headers = {"Authorization": f"Basic {basic}", "Content-Type": "application/x-www-form-urlencoded"}
    data = {"grant_type": "refresh_token", "refresh_token": refresh_token}

    resp = requests.post(token_url, headers=headers, data=data, timeout=timeout_s)
    payload = resp.json() if resp.headers.get("Content-Type","").startswith("application/json") else {"raw": resp.text}

    if resp.status_code != 200:
        raise SpotifyAuthError(f"Refresh failed ({resp.status_code}): {payload}")

    access_token = payload["access_token"]
    expires_in = int(payload.get("expires_in", 3600))
    now = int(time.time())
    expires_at = now + expires_in - 30
    new_refresh_token = payload.get("refresh_token") or refresh_token
    return {"access_token": access_token, "refresh_token": new_refresh_token, "expires_at": expires_at}

def get_valid_access_token(client_id: str, client_secret: str, refresh_token_fallback: str) -> dict:
    state = load_token_state()
    if not state.get("refresh_token"):
        state["refresh_token"] = refresh_token_fallback

    now = int(time.time())
    if state.get("access_token") and now < int(state.get("expires_at", 0)):
        return state

    refreshed = refresh_spotify_access_token(state["refresh_token"], client_id, client_secret)
    state.update(refreshed)
    save_token_state(state)
    return state

def spotify_api_call(
    method: str,
    url: str,
    client_id: str,
    client_secret: str,
    refresh_token_fallback: str,
    params: dict | None = None,
    json_body: dict | None = None,
    timeout_s: int = 20,
    max_429_retries: int = 5,
    run_id: str | None = None
):
    run_id = run_id or str(uuid.uuid4())
    state = get_valid_access_token(client_id, client_secret, refresh_token_fallback)

    def _req(access_token: str):
        headers = {"Authorization": f"Bearer {access_token}", "Accept": "application/json"}
        return requests.request(method, url, headers=headers, params=params, json=json_body, timeout=timeout_s)

    attempt = 0
    while True:
        attempt += 1
        resp = _req(state["access_token"])

        # salvezi raw indiferent de rezultat
        raw_row = {
            "run_id": run_id,
            "token_name": TOKEN_NAME,
            "method": method.upper(),
            "request_url": resp.request.url,
            "request_params_json": json.dumps(params or {}, ensure_ascii=False),
            "request_body_json": json.dumps(json_body or {}, ensure_ascii=False),
            "http_status": int(resp.status_code),
            "response_headers_json": json.dumps(dict(resp.headers), ensure_ascii=False),
            "payload_json": resp.text,
            "attempt": attempt,
            "request_hash": request_hash(method, url, params, json_body, TOKEN_NAME)
        }
        save_raw_api_call(raw_row)

        # 401 -> refresh o singură dată
        if resp.status_code == 401 and attempt == 1:
            refreshed = refresh_spotify_access_token(state["refresh_token"], client_id, client_secret)
            state.update(refreshed)
            save_token_state(state)
            continue

        # 429 -> respectă Retry-After
        if resp.status_code == 429 and attempt <= max_429_retries:
            retry_after = resp.headers.get("Retry-After")
            sleep_s = int(retry_after) if retry_after and retry_after.isdigit() else min(2 ** attempt, 60)
            time.sleep(sleep_s)
            continue

        # alte erori
        if not resp.ok:
            raise RuntimeError(f"Spotify API error {resp.status_code}: {resp.text}")

        # returnezi JSON parse-at (dar ai păstrat și text brut în raw)
        return resp.json()